# 1. Project Setup
We import the libraries needed for data processing, scaling, and building the PyTorch model.

In [99]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

# Data Source
The data is sourced from the following link:
[Breast Cancer Dataset](https://github.com/gscdit/Breast-Cancer-Detection/blob/master/data.csv)

# 2. Data Loading
We load the Breast Cancer dataset from the CSV file and check its structure.

In [100]:
df = pd.read_csv('/kaggle/input/datasets/mehedimeraj/breast-c-dataset-pytorch/data (1).csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

# 3. Data Cleaning
We remove the **id** column and other unneeded columns because they do not help the model learn patterns.

In [101]:
df.drop(columns = ['id'], inplace = True)

# 4. Split and Scale
We split the data into training and testing sets. We use a **Standard Scaler** to make the data easier for the neuron to process.

In [102]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:31], df.iloc[:,0], test_size = 0.2)

In [103]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 5. Label Encoding
We convert the diagnosis labels into numbers 1 and 0 so the model can calculate the loss.

In [104]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [105]:
y_train

array([1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1,
       1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1,
       1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0,
       0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0,
       1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0,
       1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1,
       1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0,
       1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0,
       1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1,
       0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1,
       1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0,

# 6. Tensor Conversion
We move the data from NumPy arrays to PyTorch tensors to allow gradient tracking.

In [106]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)
print(X_train_tensor.shape)
print(X_test_tensor.shape)

torch.Size([455, 30])
torch.Size([114, 30])


# 7. Model Architecture
We define a single neuron network. It uses the Binary Cross Entropy loss formula:

$$L = -(y \log(y_{pred}) + (1 - y) \log(1 - y_{pred}))$$

**Note:** We must use **.mean()** in the loss function to get a single scalar value. This is required for autograd to start calculating derivatives.

In [107]:
class OneNeuronNet:
    def __init__(self, n_inputs, n_neurons):
        self.weights = torch.randn(n_inputs, n_neurons, dtype = torch.float64, requires_grad = True)
        self.biases = torch.zeros(1,n_neurons, dtype = torch.float64, requires_grad= True)

    def forward_pass(self, inputs):
        self.output = torch.matmul(inputs, self.weights) + self.biases
        y_pred = torch.sigmoid(self.output)
        return y_pred

    def binary_cross_entropy_loss(self,y,y_pred):
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, 1e-7, 1 - 1e-7)
        return -(y * torch.log(y_pred) + (1-y) * torch.log(1-y_pred)).mean() # autograd needs a scalar value to start for finding derivatives

# 8. Important Parameters

In [109]:
learning_rate = 0.1

epochs = 25

# 8. Training Loop
In this loop, we use **.view(-1, 1)** on our labels to align shapes. This is critical to prevent broadcasting errors that create huge, incorrect matrices. We also use in place updates like **-=** to keep our weights as leaf tensors.

In [110]:
model = OneNeuronNet(X_train_tensor.shape[1], 1)

for epoch in range(epochs):
    #forward pass
    y_pred = model.forward_pass(X_train_tensor)

    #loss calculation
    y_train_tensor = y_train_tensor.view(-1,1) ##converting from (455,) to (455,1) so that it doesnt broadcast into (114,114) tensor
    loss = model.binary_cross_entropy_loss(y_train_tensor, y_pred)

    #backward_pass
    loss.backward()

    #update weights and bias
    with torch.no_grad():
        model.weights -= learning_rate * model.weights.grad # without in-place operations it will create non-leaf tensor which will cause error.  
        model.biases -= learning_rate * model.biases.grad

    model.weights.grad.zero_()
    model.biases.grad.zero_()

    print(f"Epoch {epoch+1}: {loss.item()}")
    

Epoch 1: 2.1316183478450608
Epoch 2: 1.7956089745603285
Epoch 3: 1.5204129184214972
Epoch 4: 1.2990251548113554
Epoch 5: 1.1211474003613744
Epoch 6: 0.9762381449406996
Epoch 7: 0.8577472790562412
Epoch 8: 0.7615087357449778
Epoch 9: 0.6834046724649517
Epoch 10: 0.6196656883664885
Epoch 11: 0.5672884921816952
Epoch 12: 0.5238011985580315
Epoch 13: 0.4872783875757106
Epoch 14: 0.45630689476060604
Epoch 15: 0.4298392338840871
Epoch 16: 0.40706619025408375
Epoch 17: 0.3873387746167573
Epoch 18: 0.370126657494233
Epoch 19: 0.35499608654036385
Epoch 20: 0.3415948498089966
Epoch 21: 0.32963874826953216
Epoch 22: 0.3188988159682709
Epoch 23: 0.30919001219639075
Epoch 24: 0.3003618822748807
Epoch 25: 0.29229118343025795


# 9. Final Evaluation
We check the model accuracy on the test set. We use a **0.7 threshold** to turn probabilities into binary classes and match the label shapes again for a correct final score.

In [116]:
with torch.no_grad():
    y_pred = model.forward_pass(X_test_tensor)
    y_pred = (y_pred > 0.7).float()
    y_test_tensor = y_test_tensor.view(-1,1) #converting from (114,) to (114,1) so that it doesnt broadcast into (114,114) tensor
    accuracy = (y_pred == y_test_tensor).float().mean()
    print(f"accuracy: {accuracy.item()}")

accuracy: 0.9122806787490845
